In [ ]:
# ===== Cell 1: Install, load, rebuild train split, inspect attack columns =====
!pip install -q sdv

import os, glob, warnings, random, time
import numpy as np
import pandas as pd
import torch
warnings.filterwarnings("ignore")

# ---- Config (locked before any results) ----
EPOCHS_LIST = [300, 500, 1000]
SEEDS       = [42, 45, 50]
N_SYN       = 3300
OUT_DIR     = "/content/ctgan_fullschema"
os.makedirs(OUT_DIR, exist_ok=True)

def find_file(*names):
    for n in names:
        hits = glob.glob(f"/content/**/{glob.escape(n)}", recursive=True)
        if hits: return hits[0]
    raise FileNotFoundError(f"none of {names} found")

FINAL_CSV = find_file("Final Data.csv", "Final Data.csv", "Final_Data.csv")
TEST_CSV  = find_file("test.csv", "test.csv")
print("final:", FINAL_CSV, "\ntest :", TEST_CSV)

final = pd.read_csv(FINAL_CSV, low_memory=False)
test  = pd.read_csv(TEST_CSV,  low_memory=False)[final.columns]   # align column order

# train = Final minus test (split was grouped by flow_key)
train = final[~final['flow_key'].isin(test['flow_key'])].reset_index(drop=True)
attack_train = train[train['is_attack'] == True].drop(columns=['is_attack']).reset_index(drop=True)
benign_train = train[train['is_attack'] == False].reset_index(drop=True)

print("\ntrain:", train.shape, train['is_attack'].value_counts().to_dict())
print("test :", test.shape,  test['is_attack'].value_counts().to_dict())
assert len(attack_train) == 3116 and len(benign_train) == 404, "train split does not match earlier runs"
assert len(train) + len(test) == len(final)

# ---- What CTGAN will have to learn on the full schema ----
all_nan  = [c for c in attack_train.columns if attack_train[c].isna().all()]
constant = [c for c in attack_train.columns
            if c not in all_nan and attack_train[c].nunique(dropna=False) == 1]
partial_nan = [c for c in attack_train.columns
               if c not in all_nan and attack_train[c].isna().any()]

print(f"\nAttack-train columns: {attack_train.shape[1]}")
print(f"  all-NaN (passed through as NaN): {len(all_nan)}")
print(f"  constant (CTGAN must learn):     {len(constant)}")
print(f"  partially NaN (NaN pattern):     {len(partial_nan)}")
print("\nall-NaN:", all_nan)
print("\nconstant:", constant)
print("\nGPU available:", torch.cuda.is_available())

In [ ]:
# ===== Cell 2: Metadata + full-schema CTGAN train function + smoke test =====
from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer
from rdt.transformers.numerical import FloatFormatter

KEY_PARTS = ['timeframe_id', 'proto', 'ip_a', 'ip_b', 'port_a', 'port_b']

# Training frame: everything except flow_key (rebuilt) and the 4 all-NaN columns (passed through)
gan_df = attack_train.drop(columns=['flow_key'] + all_nan).copy()
orig_dtypes = attack_train.dtypes

def sdtype_for(c):
    s = gan_df[c]
    if s.dtype == object or s.nunique(dropna=True) <= 10:   # strings + low-cardinality (incl. constants)
        return {'sdtype': 'categorical'}
    return {'sdtype': 'numerical', 'computer_representation': 'Float'}

cols = {c: sdtype_for(c) for c in gan_df.columns}
metadata = Metadata.load_from_dict({'tables': {'attack': {'columns': cols}}})
num_nan_cols = [c for c in partial_nan if c in gan_df.columns and cols[c]['sdtype'] == 'numerical']

n_cat = sum(v['sdtype'] == 'categorical' for v in cols.values())
print(f"CTGAN models {gan_df.shape[1]} columns: {n_cat} categorical, {gan_df.shape[1]-n_cat} numerical")
print(f"NaN-pattern columns (from_column): {len(num_nan_cols)}")

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def train_ctgan(epochs, seed, n=N_SYN, save=True):
    set_seed(seed)
    synth = CTGANSynthesizer(metadata, enforce_min_max_values=True, enforce_rounding=True,
                             epochs=epochs, batch_size=500, pac=10, embedding_dim=128,
                             generator_dim=(256, 256), discriminator_dim=(256, 256),
                             generator_lr=2e-4, discriminator_lr=2e-4,
                             generator_decay=1e-6, discriminator_decay=1e-6,
                             discriminator_steps=1, log_frequency=True, verbose=False,
                             cuda=torch.cuda.is_available())
    synth.auto_assign_transformers(gan_df)
    if num_nan_cols:
        synth.update_transformers({c: FloatFormatter(missing_value_generation='from_column',
                                   learn_rounding_scheme=True, enforce_min_max_values=True)
                                   for c in num_nan_cols})
    t0 = time.time(); synth.fit(gan_df); train_s = time.time() - t0

    syn = synth.sample(num_rows=n)
    for c in all_nan: syn[c] = np.nan                                  # structurally empty
    for c in orig_dtypes.index:                                        # ints stay ints
        if c in syn.columns and orig_dtypes[c].kind in 'iu' and syn[c].notna().all():
            syn[c] = syn[c].round().astype(orig_dtypes[c])
    syn['flow_key'] = syn[KEY_PARTS].astype(str).agg('|'.join, axis=1)
    syn['is_attack'] = True
    syn = syn[final.columns]

    if save:
        tag = f"e{epochs}_s{seed}"
        syn.to_csv(f"{OUT_DIR}/ctgan_full_{tag}.csv", index=False)
        try: synth.get_loss_values().to_csv(f"{OUT_DIR}/loss_{tag}.csv", index=False)
        except Exception as e: print("loss not saved:", e)
    return syn, train_s

# ---- Smoke test: 2 epochs, 200 rows, not saved ----
smoke, t = train_ctgan(epochs=2, seed=42, n=200, save=False)
print(f"\nsmoke: {smoke.shape} in {t:.1f}s | columns match: {list(smoke.columns) == list(final.columns)}")
print("all-NaN cols still NaN:", all(smoke[c].isna().all() for c in all_nan))
exact = sum((smoke[c] == attack_train[c].iloc[0]).all() for c in constant)
print(f"constants reproduced exactly after 2 epochs: {exact}/{len(constant)}  (low is expected here)")

In [ ]:
# ===== Cell 3: 9 full-schema CTGAN runs (epochs x seeds), resumable =====
import gc

# Optional: also copy each output to Google Drive so a disconnect doesn't lose it
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/ctgan_fullschema'; os.makedirs(DRIVE_DIR, exist_ok=True)

log_rows = []
for epochs in EPOCHS_LIST:
    for seed in SEEDS:
        tag  = f"e{epochs}_s{seed}"
        path = f"{OUT_DIR}/ctgan_full_{tag}.csv"
        if os.path.exists(path):
            print(f"[skip] {tag} already saved"); syn = pd.read_csv(path, low_memory=False); train_s = np.nan
        else:
            print(f"[run ] {tag} ...", end=" ", flush=True)
            syn, train_s = train_ctgan(epochs, seed)
            print(f"done in {train_s/60:.1f} min")
            if USE_DRIVE:
                import shutil; shutil.copy(path, DRIVE_DIR)
                lp = f"{OUT_DIR}/loss_{tag}.csv"
                if os.path.exists(lp): shutil.copy(lp, DRIVE_DIR)

        # quick sanity checks on the generated rows
        const_ok = sum((syn[c] == attack_train[c].iloc[0]).all() for c in constant)
        log_rows.append({
            'epochs': epochs, 'seed': seed, 'rows': len(syn), 'train_min': round(train_s/60, 2),
            'dups': int(syn.drop(columns=['flow_key']).duplicated().sum()),
            'NaN_cells': int(syn.isna().sum().sum()),
            'constants_exact': f"{const_ok}/{len(constant)}",
            'timeframes': sorted(syn['timeframe_id'].unique().tolist()),
        })
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

run_log = pd.DataFrame(log_rows)
run_log.to_csv(f"{OUT_DIR}/run_log.csv", index=False)
print("\n", run_log.to_string(index=False))
print(f"\nReal attack_train NaN cells for reference: {int(attack_train.isna().sum().sum())} "
      f"(per 3,300 rows ≈ {int(attack_train.isna().sum().sum() * 3300 / len(attack_train))})")

In [ ]:
import os, glob, shutil
from google.colab import files

SRC = '/content/ctgan_fullschema'          # where train_ctgan() saved the runs
print(sorted(os.listdir(SRC)))             # expect 9 ctgan_full_e*_s*.csv + 9 loss_*.csv (+ run_log.csv)

run_log.to_csv(f'{SRC}/run_log.csv', index=False)
shutil.make_archive('/content/ctgan_fullschema', 'zip', SRC)
files.download('/content/ctgan_fullschema.zip')

In [ ]:
# ===== Eval Cell 1: TSTR-HARD for the 9 full-schema CTGAN runs (same pipeline as Table I) =====
import os, glob, numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix

SEED = 42
SRC  = '/content/ctgan_fullschema'

# --- same train/test split as before ---
test = pd.read_csv('/content/test.csv', low_memory=False)
if os.path.exists('/content/train.csv'):
    train = pd.read_csv('/content/train.csv', low_memory=False)
else:   # rebuild: Final minus test (split grouped by flow_key)
    final = pd.read_csv(glob.glob('/content/Final*Data*.csv')[0], low_memory=False)
    train = final[~final['flow_key'].isin(test['flow_key'])].reset_index(drop=True)
test = test[train.columns]
atk_tr = train[train['is_attack'] == True].drop(columns=['is_attack']).reset_index(drop=True)
ben_tr = train[train['is_attack'] == False].reset_index(drop=True)
assert len(atk_tr) == 3116 and len(ben_tr) == 404, (len(atk_tr), len(ben_tr))

ID_COLS = ['flow_key', 'timeframe_id', 'ip_a', 'ip_b', 'port_a', 'port_b']
CONTEXT_PREFIX = ('source_', 'target_service_', 'target_ip_', 'flow_share_of_')
MODELS = ['RandomForest', 'LogisticReg', 'SVM', 'MLP']

def load_syn(df):
    s = df.reindex(columns=train.columns).drop(columns=['is_attack'])
    for c in s.columns:
        if c in ID_COLS: continue
        if train[c].dtype == bool:
            s[c] = s[c].astype(str).str.strip().str.lower().map({'true':1,'false':0,'1':1,'0':0,'1.0':1,'0.0':0})
        else:
            s[c] = pd.to_numeric(s[c], errors='coerce')
    return s

def prep_full(df):
    X = df.drop(columns=[c for c in ID_COLS + ['is_attack'] if c in df.columns]).copy()
    for c in X.select_dtypes('bool').columns: X[c] = X[c].astype(int)
    return X

def prep_hard(df):
    X = prep_full(df)
    return X.drop(columns=[c for c in X.columns if c.startswith(CONTEXT_PREFIX)])

def make_models():
    return {
        'RandomForest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=SEED, n_jobs=-1),
        'LogisticReg':  LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED),
        'SVM':          LinearSVC(C=1.0, class_weight='balanced', max_iter=5000, random_state=SEED),
        'MLP':          MLPClassifier(hidden_layer_sizes=(100,50), activation='relu', solver='adam',
                                      early_stopping=True, max_iter=500, random_state=SEED),
    }

def evaluate(attack_rows, name, prep_fn=prep_hard):
    a = attack_rows.copy(); a['is_attack'] = True
    tr = pd.concat([a, ben_tr], ignore_index=True)
    Xtr, ytr = prep_fn(tr), tr['is_attack'].astype(int)
    Xte, yte = prep_fn(test)[Xtr.columns], test['is_attack'].astype(int)
    rows = []
    for m, clf in make_models().items():
        yp = make_pipeline(SimpleImputer(strategy='constant', fill_value=0, keep_empty_features=True),
                           StandardScaler(), clf).fit(Xtr, ytr).predict(Xte)
        tn, fp, fn, tp = confusion_matrix(yte, yp).ravel()
        rows.append({'Generator': name, 'Model': m, 'Acc': accuracy_score(yte, yp),
                     'Prec': precision_score(yte, yp, zero_division=0), 'Rec': recall_score(yte, yp),
                     'F1': f1_score(yte, yp), 'Benign_FPR': fp/(fp+tn), 'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp})
    return pd.DataFrame(rows)

# --- load the 9 runs ---
SYN = {}
for e in [300, 500, 1000]:
    for s in [42, 45, 50]:
        SYN[f'CTGAN_e{e}_s{s}'] = load_syn(pd.read_csv(f'{SRC}/ctgan_full_e{e}_s{s}.csv', low_memory=False))
print('loaded:', list(SYN), '| hard features:', prep_hard(test).shape[1])

# --- TRTR sanity + 9 runs ---
res = pd.concat([evaluate(atk_tr, 'Real (TRTR)')] + [evaluate(X, g) for g, X in SYN.items()], ignore_index=True)
res.to_csv(f'{SRC}/tstr_hard_ctgan_fullschema.csv', index=False)

ORDER = ['Real (TRTR)'] + list(SYN)
f1 = res.pivot(index='Generator', columns='Model', values='F1').reindex(ORDER)[MODELS]
f1['Mean_F1'] = f1[MODELS].mean(axis=1)
f1['Mean_FPR'] = res.groupby('Generator')['Benign_FPR'].mean().reindex(ORDER)
print('\n===== TSTR-HARD F1 per run =====');  print(f1.round(3).to_string())

# --- mean ± std per epoch setting (3 seeds) ---
f1r = f1.drop(index='Real (TRTR)').copy()
f1r['epochs'] = [int(i.split('_e')[1].split('_')[0]) for i in f1r.index]
agg = f1r.groupby('epochs')[MODELS + ['Mean_F1']].agg(['mean', 'std'])
print('\n===== Mean ± std over seeds 42/45/50 =====')
print(pd.DataFrame({c: agg[c]['mean'].round(3).astype(str) + ' ± ' + agg[c]['std'].round(3).astype(str)
                    for c in MODELS + ['Mean_F1']}).to_string())
print('\nReference: old CTGAN (1000 ep, constants dropped, seed 42) mean F1 = 0.968; TRTR = 0.994')

In [ ]:
# ===== Eval Cell 2: Fidelity KS / JSD / MMD² (HARD) for the 9 CTGAN runs =====
from scipy.stats import ks_2samp
from scipy.spatial.distance import jensenshannon
from sklearn.preprocessing import QuantileTransformer
from sklearn.metrics.pairwise import euclidean_distances

def ks_jsd_all(R, X, bins=20):
    ks, js = [], []
    for c in R.columns:
        r, x = R[c].values, X[c].values
        if np.ptp(np.r_[r, x]) == 0:
            ks.append(0.0); js.append(0.0); continue
        ks.append(ks_2samp(r, x).statistic)
        edges = np.unique(np.quantile(np.r_[r, x], np.linspace(0, 1, bins + 1)))
        if len(edges) < 2: edges = np.array([edges[0] - 0.5, edges[0] + 0.5])
        pr, _ = np.histogram(r, edges); px, _ = np.histogram(x, edges)
        js.append(jensenshannon(pr + 1e-12, px + 1e-12, base=2) ** 2)
    return np.mean(ks), np.mean(js)

def subs(A, n, seed): return A[np.random.default_rng(seed).choice(len(A), min(n, len(A)), replace=False)]

test_att = test[test['is_attack'] == True].drop(columns=['is_attack'])
R  = prep_hard(test_att).fillna(0).astype(float)
qt = QuantileTransformer(n_quantiles=1000, output_distribution='uniform', random_state=SEED).fit(R)
Rz = qt.transform(R)
d0 = euclidean_distances(subs(Rz, 1000, 0), squared=True)
gams = [1.0 / (np.median(d0[d0 > 0]) * f) for f in (0.1, 1, 10)]
Kf = lambda P, Q: sum(np.exp(-g * euclidean_distances(P, Q, squared=True)) for g in gams) / len(gams)

def mmd2(Xz, n=1000):
    A, B = subs(Rz, n, 1), subs(Xz, n, 2); m, k = len(A), len(B)
    Kxx, Kyy, Kxy = Kf(A, A), Kf(B, B), Kf(A, B)
    return (Kxx.sum() - np.trace(Kxx)) / (m*(m-1)) + (Kyy.sum() - np.trace(Kyy)) / (k*(k-1)) - 2 * Kxy.mean()

FID_SETS = {'Real-train (ref)': atk_tr, **SYN}
rows = []
for g, df in FID_SETS.items():
    X = prep_hard(df)[R.columns].fillna(0).astype(float)
    ks, js = ks_jsd_all(R, X)
    rows.append({'Generator': g, 'n_rows': len(X), 'KS': ks, 'JSD': js, 'MMD2': mmd2(qt.transform(X))})
fid = pd.DataFrame(rows).set_index('Generator')
fid.to_csv(f'{SRC}/fidelity_hard_ctgan_fullschema.csv')
print(f'reference: real test attacks {R.shape}')
print('\n===== Fidelity per run (HARD, lower = better) =====')
print(fid.round(4).to_string())

# --- mean ± std per epoch setting ---
fr = fid.drop(index='Real-train (ref)').copy()
fr['epochs'] = [int(i.split('_e')[1].split('_')[0]) for i in fr.index]
agg = fr.groupby('epochs')[['KS', 'JSD', 'MMD2']].agg(['mean', 'std'])
print('\n===== Mean ± std over seeds 42/45/50 =====')
print(pd.DataFrame({c: agg[c]['mean'].round(4).astype(str) + ' ± ' + agg[c]['std'].round(4).astype(str)
                    for c in ['KS', 'JSD', 'MMD2']}).to_string())
print('\nReference (Table II, hard): old CTGAN KS 0.146 | JSD 0.130 | MMD² 0.100 ; Real-train 0.009 | 0.001 | 0.000')